# Binning V2 Test

Test the new join_asof-based binning implementation.


In [ ]:
import polars as pl
from datetime import date, datetime
from okx.store import OrderbookStore
from okx.features import generate_time_bins, bin_ob_v2, bin_by_times

# Initialize store
store = OrderbookStore(
    data_root='data/okx',
    manifest_path='data/okx/manifest.sqlite'
)


## Test 1: Generate time bins


In [ ]:
# Test time bin generation
start_ms = int(datetime(2025, 9, 1, 0, 0, 0).timestamp() * 1000)
end_ms = int(datetime(2025, 9, 1, 1, 0, 0).timestamp() * 1000)

bins_5m = generate_time_bins(start_ms, end_ms, '5m', include_next=False)
print(f"Generated {len(bins_5m)} bins for 1 hour at 5m intervals")
print(f"First bin: {datetime.fromtimestamp(bins_5m[0]/1000)}")
print(f"Last bin: {datetime.fromtimestamp(bins_5m[-1]/1000)}")

bins_5m_extended = generate_time_bins(start_ms, end_ms, '5m', include_next=True)
print(f"\nWith include_next: {len(bins_5m_extended)} bins")
print(f"Last bin: {datetime.fromtimestamp(bins_5m_extended[-1]/1000)}")


## Test 2: Compare old vs new binning


In [ ]:
import time

test_date = date(2025, 9, 1)

# Old binning (group_by_dynamic)
print("Testing old binning with group_by_dynamic...")
t0 = time.time()
lf_old = store.get(
    inst_family='BTC-USD',
    inst_type='SWAP',
    dates=[test_date],
    depth=1,
    binning='5m',
    features=['trim', 'bin_ff', 'sink_bins']
)
df_old = lf_old.collect()
t_old = time.time() - t0
print(f"Old: {t_old:.3f}s, {len(df_old)} rows\n")

# New binning (join_asof)
print("Testing new binning with join_asof...")
t0 = time.time()
lf_new = store.get(
    inst_family='BTC-USD',
    inst_type='SWAP',
    dates=[test_date],
    depth=1,
    binning='5m',
    features=['trim', 'bin_v2_ff', 'sink_bins']
)
df_new = lf_new.collect()
t_new = time.time() - t0
print(f"New: {t_new:.3f}s, {len(df_new)} rows")
print(f"\nSpeedup: {t_old/t_new:.2f}x")
print(f"Results match: {len(df_old) == len(df_new)}")
